In [24]:
from pyspark.sql import SparkSession

sparkSession = SparkSession.builder.appName("FootballPlaystyleAnalysis").master("local[*]").getOrCreate()
spark = sparkSession
sparkSession

Veri Setlerini Okuma

In [25]:
events = spark.read.option("multiline", "true").json("statsbomb-open-data/data/events/*.json")

26/07/10 18:08:21 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: statsbomb-open-data/data/events/*.json.
java.io.FileNotFoundException: File statsbomb-open-data/data/events/*.json does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apac

In [26]:
##Events kontrolleri
events.printSchema()

events.count()

len(events.columns)

events.show(5, truncate=False)

root
 |-- 50_50: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- bad_behaviour: struct (nullable = true)
 |    |-- card: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_receipt: struct (nullable = true)
 |    |-- outcome: struct (nullable = true)
 |    |    |-- id: long (nullable = true)
 |    |    |-- name: string (nullable = true)
 |-- ball_recovery: struct (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- recovery_failure: boolean (nullable = true)
 |-- block: struct (nullable = true)
 |    |-- deflection: boolean (nullable = true)
 |    |-- offensive: boolean (nullable = true)
 |    |-- save_block: boolean (nullable = true)
 |-- carry: struct (nullable = true)
 |    |-- end_location: array (nullable = true)
 |    |    |-- element: double (containsNull = true)
 |-- clearan

+-----+-------------+------------+-------------+-----+-----+---------+------------+-------+----+--------+--------------+--------+----------+--------+----------+------------------------------------+-----+---------------+------------+------------+------+----------+----------+----+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+------+------------------+-------------------------+----------+-------------------------------+----------+---------------+--------------------------------------+------+----+------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
##event types
from pyspark.sql.functions import col

events.select(col("type.name").alias("event_type")) \
      .distinct() \
      .orderBy("event_type") \
      .show(100, truncate=False)

+-----------------+
|event_type       |
+-----------------+
|50/50            |
|Bad Behaviour    |
|Ball Receipt*    |
|Ball Recovery    |
|Block            |
|Camera On        |
|Camera off       |
|Carry            |
|Clearance        |
|Dispossessed     |
|Dribble          |
|Dribbled Past    |
|Duel             |
|Error            |
|Foul Committed   |
|Foul Won         |
|Goal Keeper      |
|Half End         |
|Half Start       |
|Injury Stoppage  |
|Interception     |
|Miscontrol       |
|Offside          |
|Own Goal Against |
|Own Goal For     |
|Pass             |
|Player Off       |
|Player On        |
|Pressure         |
|Referee Ball-Drop|
|Shield           |
|Shot             |
|Starting XI      |
|Substitution     |
|Tactical Shift   |
+-----------------+



In [28]:
events.groupBy(
    col("type.name").alias("event_type")
).count().orderBy(col("count").desc()).show(100, truncate=False)

+-----------------+-------+
|event_type       |count  |
+-----------------+-------+
|Pass             |4103347|
|Ball Receipt*    |3844869|
|Carry            |3204878|
|Pressure         |1394727|
|Ball Recovery    |461754 |
|Duel             |310469 |
|Clearance        |196297 |
|Block            |168752 |
|Dribble          |146663 |
|Goal Keeper      |132397 |
|Miscontrol       |126295 |
|Foul Committed   |117581 |
|Foul Won         |111772 |
|Dispossessed     |110366 |
|Shot             |108282 |
|Interception     |96130  |
|Dribbled Past    |90736  |
|Substitution     |28044  |
|Injury Stoppage  |18172  |
|Half End         |17216  |
|Half Start       |17216  |
|50/50            |15838  |
|Tactical Shift   |11739  |
|Starting XI      |8470   |
|Referee Ball-Drop|6264   |
|Shield           |6007   |
|Player Off       |4541   |
|Player On        |4500   |
|Bad Behaviour    |2987   |
|Camera On        |2595   |
|Error            |2256   |
|Offside          |1513   |
|Camera off       |6

In [29]:
passes = events.filter(col("type.name") == "Pass")
passes.count()

4103347

In [30]:
## Extract the necessary columns for analysis
passes = passes.select(
    col("player.id").alias("passer_id"),
    col("player.name").alias("passer"),
    col("pass.recipient.id").alias("receiver_id"),
    col("pass.recipient.name").alias("receiver"),
    col("team.id").alias("team_id"),
    col("team.name").alias("team"),
    "minute",
    "second",
    "location",
    col("pass.end_location").alias("end_location")
)

##Show
passes.show(10, truncate=False)

+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|passer_id|passer                   |receiver_id|receiver                 |team_id|team  |minute|second|location    |end_location|
+---------+-------------------------+-----------+-------------------------+-------+------+------+------+------------+------------+
|5487     |Antoine Griezmann        |10481      |Aurélien Djani Tchouaméni|771    |France|0     |0     |[60.0, 40.0]|[48.4, 38.1]|
|10481    |Aurélien Djani Tchouaméni|24778      |Eduardo Camavinga        |771    |France|0     |2     |[47.9, 37.4]|[49.3, 28.7]|
|24778    |Eduardo Camavinga        |8519       |Dayotchanculle Upamecano |771    |France|0     |4     |[49.0, 25.2]|[38.1, 46.8]|
|8519     |Dayotchanculle Upamecano |3961       |N'Golo Kanté             |771    |France|0     |7     |[41.6, 49.4]|[49.5, 52.3]|
|3961     |N'Golo Kanté             |17592      |William Saliba           |771    |

In [31]:
passes.filter(col("receiver").isNull()).count()

256163

In [32]:
##Filtreyi uygula - alıcısı olmayan pasları çıkar
passes = passes.filter(col("receiver").isNotNull())
## Sayı kontrol
passes.count()

3847184

In [33]:
## create an edge list for the passes
edges = (
    passes.groupBy(
        "passer_id",
        "passer",
        "receiver_id",
        "receiver",
        "team_id",
        "team"
    )
    .count()
    .withColumnRenamed("count", "pass_count")

    
    
)

##kontrol
edges.show(20, truncate=False)

edges.count()

+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|passer_id|passer                       |receiver_id|receiver                      |team_id|team               |pass_count|
+---------+-----------------------------+-----------+------------------------------+-------+-------------------+----------+
|8519     |Dayotchanculle Upamecano     |4445       |Jules Koundé                  |771    |France             |77        |
|5204     |Bruno Miguel Borges Fernandes|41092      |Nuno Mendes                   |780    |Portugal           |39        |
|3961     |N'Golo Kanté                 |3009       |Kylian Mbappé Lottin          |771    |France             |21        |
|10595    |Raphael Dias Belloli         |3063       |Danilo Luiz da Silva          |781    |Brazil             |20        |
|30486    |Pedro González López         |5203       |Sergio Busquets i Burgos      |772    |Spain              |60        |
|16022  

208518

In [34]:
## creating verteices

vertices = (
    passes.select(
        col("passer_id").alias("id"),
        col("passer").alias("name"),
        "team_id",
        "team"
    )
    .distinct()
)

vertices.count()
vertices.show(20, truncate=False)

+-----+--------------------------------+-------+------------------------+
|id   |name                            |team_id|team                    |
+-----+--------------------------------+-------+------------------------+
|3009 |Kylian Mbappé Lottin            |131    |Paris Saint-Germain     |
|6840 |Marcos Llorente Moreno          |772    |Spain                   |
|23725|Roman Bezus                     |911    |Ukraine                 |
|5487 |Antoine Griezmann               |212    |Atlético Madrid         |
|34639|Vitor Machado Ferreira          |131    |Paris Saint-Germain     |
|48396|Rocco Reitz                     |185    |Borussia Mönchengladbach|
|13620|Éder Gabriel Militão            |781    |Brazil                  |
|25305|Pedro Guilherme Abreu dos Santos|781    |Brazil                  |
|18618|Serhiy Kryvtsov                 |911    |Ukraine                 |
|31900|Oleksandr Karavaev              |911    |Ukraine                 |
|11396|Florian Grillitsch             